# Drug Classification

**Tabular Classification Project** — Predict the best drug for a patient.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Drug200 (200 rows × 6 columns, Kaggle)  
Target: `Drug` (5 classes: drugA, drugB, drugC, drugX, drugY)

## 2 · Project Overview

We predict which **drug** a doctor should prescribe to a patient based on their age, sex, blood pressure, cholesterol level, and sodium-to-potassium ratio. This is a **5-class classification** task on a small medical dataset.

## 3 · Learning Objectives

1. Handle multi-class classification with ordinal and nominal features.
2. Encode mixed feature types (ordinal BP/Cholesterol, nominal Sex).
3. Compare boosting models on small multi-class datasets.
4. Evaluate with weighted F1, per-class metrics, and confusion matrix.

## 4 · Problem Statement

Given a patient's **Age, Sex, Blood Pressure, Cholesterol, and Na_to_K ratio**, predict the most appropriate **Drug** from 5 options.

## 5 · Why This Project Matters

- Clinical decision support systems help doctors choose optimal treatments.
- Multi-class classification with ordinal features is common in healthcare.
- Small datasets require careful preprocessing and evaluation.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 200 |
| **Columns** | 6 |
| **Features** | Age, Sex, BP, Cholesterol, Na_to_K |
| **Target** | Drug (5 classes) |
| **Source** | Kaggle `prathamtripathi/drug-classification` |

## 7 · Dataset Source and License Notes

- **Kaggle**: `prathamtripathi/drug-classification`
- **License**: CC0 Public Domain.
- **Limitations**: Synthetic/simplified medical data — not for real clinical use.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, i in [("catboost",None),("lightgbm",None),("xgboost",None),("flaml",None),("lazypredict",None)]:
    _install(p, i)
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, re, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
np.random.seed(SEED)
print("Core imports loaded.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Drug"
TEST_SIZE = 0.2
KAGGLE_SLUG = "prathamtripathi/drug-classification"
print(f"Target: {TARGET}")

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub, glob
path = kagglehub.dataset_download(KAGGLE_SLUG)
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(df.head())

## 12 · Data Validation Checks

In [ ]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nTarget classes: {df[TARGET].unique()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["Age"].hist(bins=20, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution")

df["Na_to_K"].hist(bins=20, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Na_to_K Distribution")

df[TARGET].value_counts().plot.bar(ax=axes[2], color="steelblue")
axes[2].set_title("Drug Distribution")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

# Categorical features
for c in ["Sex", "BP", "Cholesterol"]:
    print(f"\n{c}:\n{df[c].value_counts()}")

## 14 · Target Analysis

In [ ]:
print(f"Target distribution:\n{df[TARGET].value_counts()}")
print(f"\nProportions:\n{df[TARGET].value_counts(normalize=True)}")

## 15 · Train/Validation/Test Split Strategy

In [ ]:
# Encode categorical features
le_dict = {}
for c in ["Sex", "BP", "Cholesterol"]:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    le_dict[c] = le

# Encode target
le_target = LabelEncoder()
df[TARGET] = le_target.fit_transform(df[TARGET])
print(f"Classes: {le_target.classes_}")

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Sanitize column names
X_train.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_train.columns]
X_test.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_test.columns]

## 16 · Preprocessing Strategy

Categorical features (Sex, BP, Cholesterol) are label-encoded. All features are already numeric after encoding. No scaling needed for tree models.

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part["Age_Na_to_K"] = df_part["Age"] * df_part["Na_to_K"]
    df_part["BP_Cholesterol"] = df_part["BP"] * df_part["Cholesterol"]
print(f"Features: {X_train.shape[1]}")

## 18 · Baseline Model

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
bl_pred = baseline.predict(X_test)
print("Baseline — Logistic Regression:")
print(f"  Accuracy: {accuracy_score(y_test, bl_pred):.4f}")
print(f"  F1:       {f1_score(y_test, bl_pred, average='weighted'):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lp = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
lp_models, lp_preds = lp.fit(X_train, X_test, y_train, y_test)
print(lp_models.head(20))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task="classification", time_budget=60,
               metric="accuracy", verbose=0, seed=SEED)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy : {accuracy_score(y_test, flaml_pred):.4f}")
    print(f"  F1       : {f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## 21 · Additional Best-Library Workflow

## 22 · Top 3 Model Selection

## 23 · Final Training and Evaluation of Top 3

In [ ]:
import torch
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

device = "gpu" if torch.cuda.is_available() else "cpu"
xgb_device = "cuda" if torch.cuda.is_available() else "cpu"

n_classes = len(np.unique(y_train))
if n_classes == 2:
    n_pos = int(y_train.sum()); n_neg = int(len(y_train) - n_pos)
    spw = n_neg / max(n_pos, 1)
else:
    spw = 1.0

models = {
    "CatBoost": CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                                    auto_class_weights="Balanced",
                                    task_type="GPU" if device=="gpu" else "CPU",
                                    verbose=0, random_seed=SEED),
    "LightGBM": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                    is_unbalance=True, device=device,
                                    verbose=-1, random_state=SEED),
    "XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                  scale_pos_weight=spw if n_classes==2 else 1.0,
                                  device=xgb_device,
                                  eval_metric="logloss", verbosity=0, random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    results[name] = {"accuracy": acc, "f1": f1}
    print(f"{name}: Acc={acc:.4f} F1={f1:.4f}")

res_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nRanking:")
print(res_df)

## 24 · Error Analysis

In [ ]:
best_name = res_df.index[0]
best_model = models[best_name]
preds_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(y_test, preds_best))

errors = X_test.copy()
errors["true"] = y_test.values
errors["pred"] = preds_best
errors["wrong"] = errors["true"] != errors["pred"]
print(f"Total errors: {errors['wrong'].sum()} / {len(errors)} ({errors['wrong'].mean()*100:.1f}%)")
print("\nSample misclassifications:")
print(errors[errors["wrong"]].head(10))

## 25 · Interpretation and Business Insight

- **Na_to_K ratio** is the strongest predictor — it largely determines the drug choice.
- Tree models handle the mixed ordinal/nominal features well.
- For clinical use, explainability (SHAP) would be essential.

## 26 · Limitations

- Only 200 rows — high variance.
- Synthetic data — real drug selection involves many more factors.
- 5 classes with imbalance.

## 27 · How to Improve

1. Cross-validation for stability.
2. Feature importance analysis.
3. SHAP explanations.
4. Collect real clinical data.

## 28 · Production Considerations

- **Never** use ML alone for drug prescribing.
- Explainability required by healthcare regulations.
- Must be validated by domain experts.

## 29 · Common Mistakes

1. Not properly encoding ordinal features (BP has order: LOW < NORMAL < HIGH).
2. Using accuracy on imbalanced multi-class without checking per-class metrics.
3. Treating this as a real clinical dataset.

## 30 · Mini Challenge

1. Try ordinal encoding for BP and Cholesterol.
2. Create a SHAP waterfall plot.
3. Run 5-fold stratified CV.

## 31 · Final Summary

In [ ]:
metrics = {}
for name, m in results.items():
    metrics[name] = m
best = res_df.index[0]
metrics["best_model"] = best
metrics["best_accuracy"] = results[best]["accuracy"]
metrics["best_f1"] = results[best]["f1"]

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))
print("\n✅ Drug Classification notebook complete.")